# 01 · Base SECOP II congelada y catálogo institucional

- Usa la última descarga bruta **validada** (JSONL inmutable). Descargar de nuevo es opcional.
- Catálogo corregido: el **Concejo** es corporación de elección popular que ejerce control político; **no es atribuible al alcalde**.
- Verificaciones externas opcionales (otras entidades de Barrancabermeja, cobertura SECOP I 2021). Si no hay red, quedan como `NO_VERIFICADO` y las etapas siguientes lo heredan.
- Nunca modifica `datos/brutos`.

In [1]:
import json, sys, time
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
# Busca la raíz del proyecto (carpeta que contiene funciones/secop_utils.py)
for _p in [RAIZ, *RAIZ.parents][:6]:
    if (_p / "funciones" / "secop_utils.py").exists():
        RAIZ = _p
        break
else:
    raise FileNotFoundError("No se encontró funciones/secop_utils.py; abra el notebook dentro del proyecto")
sys.path.insert(0, str(RAIZ / "funciones"))
import secop_utils as su

VERSION_NB = "01.v2.0"
ETAPA = "01_base"
DESCARGAR_NUEVA = False            # True: descarga una nueva corrida desde la API (requiere internet)
VERIFICAR_COBERTURA_EXTERNA = True # intenta consultas externas; si falla, queda NO_VERIFICADO
FECHA_INICIO = "2020-01-01"
FECHA_CORTE = "2026-09-06"         # inclusive
DATASET = "jbjy-vk9h"
API = f"https://www.datos.gov.co/resource/{DATASET}.json"
RUTA_BRUTOS = RAIZ / "datos" / "brutos" / "secop_ii" / "corridas"
SALIDA = su.carpeta_etapa(RAIZ, ETAPA)
print("Proyecto:", RAIZ.name, "| salida:", su.rel(SALIDA, RAIZ))

Proyecto: PROYECT_SECOP_Bca | salida: datos/salidas/01_base


## Catálogo institucional (corregido)

In [2]:
# Grupo de atribución: solo 'Administración central' y 'Sector descentralizado vinculado' dependen del alcalde.
CATALOGO = pd.DataFrame([
    ("890201900", "Alcaldía Distrital de Barrancabermeja", "Administración central", "Administración central"),
    ("829000477", "INDERBA (deporte y recreación)", "Establecimiento público descentralizado", "Sector descentralizado vinculado"),
    ("890270833", "EDUBA (desarrollo urbano y vivienda)", "Empresa descentralizada", "Sector descentralizado vinculado"),
    ("890270948", "Inspección de Tránsito y Transporte", "Establecimiento público descentralizado", "Sector descentralizado vinculado"),
    ("829001846", "ESE Barrancabermeja", "Empresa Social del Estado distrital", "Sector descentralizado vinculado"),
    ("829001276", "Concejo de Barrancabermeja", "Corporación de elección popular (control político)", "No atribuible al alcalde"),
    ("829001855", "Personería de Barrancabermeja", "Órgano de control", "No atribuible al alcalde"),
    ("8290007456", "Contraloría de Barrancabermeja", "Órgano de control", "No atribuible al alcalde"),
    ("900136865", "Hospital Regional del Magdalena Medio", "ESE departamental", "No atribuible al alcalde"),
], columns=["nit_entidad", "entidad", "categoria_institucional", "grupo_atribucion"])
CATALOGO["es_central"] = CATALOGO["nit_entidad"].eq(su.NIT_ALCALDIA)
assert CATALOGO["nit_entidad"].is_unique and CATALOGO["es_central"].sum() == 1
assert CATALOGO.loc[CATALOGO["nit_entidad"].eq("829001276"), "grupo_atribucion"].item() == "No atribuible al alcalde"
ruta_catalogo = su.guardar_csv(CATALOGO, RAIZ / "datos" / "referencias" / "catalogo_entidades.csv")
CATALOGO

,nit_entidad,entidad,categoria_institucional,grupo_atribucion,es_central
0,890201900,Alcaldía Distrital de Barrancabermeja,Administración central,Administración central,True
1,829000477,INDERBA (deporte y recreación),Establecimiento público descentralizado,Sector descentralizado vinculado,False
2,890270833,EDUBA (desarrollo urbano y vivienda),Empresa descentralizada,Sector descentralizado vinculado,False
3,890270948,Inspección de Tránsito y Transporte,Establecimiento público descentralizado,Sector descentralizado vinculado,False
4,829001846,ESE Barrancabermeja,Empresa Social del Estado distrital,Sector descentralizado vinculado,False
5,829001276,Concejo de Barrancabermeja,Corporación de elección popular (control polít...,No atribuible al alcalde,False
6,829001855,Personería de Barrancabermeja,Órgano de control,No atribuible al alcalde,False
7,8290007456,Contraloría de Barrancabermeja,Órgano de control,No atribuible al alcalde,False
8,900136865,Hospital Regional del Magdalena Medio,ESE departamental,No atribuible al alcalde,False


## Descarga opcional (keyset por id_contrato; mismo método validado de la v1)

In [3]:
CAMPOS = [
    "nombre_entidad", "nit_entidad", "departamento", "ciudad", "orden", "proceso_de_compra",
    "id_contrato", "referencia_del_contrato", "estado_contrato", "codigo_de_categoria_principal",
    "descripcion_del_proceso", "tipo_de_contrato", "modalidad_de_contratacion",
    "justificacion_modalidad_de", "fecha_de_firma", "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato", "tipodocproveedor", "documento_proveedor", "proveedor_adjudicado",
    "es_grupo", "es_pyme", "valor_del_contrato", "valor_facturado", "valor_pagado",
    "valor_pendiente_de_ejecucion", "dias_adicionados", "urlproceso",
]


def pedir(url, params, timeout=60, intentos=3):
    seguros = ",()'"
    q = f"{url}?{urlencode(params, safe=seguros)}" if params else url
    for k in range(intentos):
        try:
            with urlopen(Request(q, headers={"Accept": "application/json"}), timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8"))
        except Exception as e:  # red o servidor
            if k == intentos - 1:
                raise RuntimeError(f"Consulta fallida: {e}") from e
            time.sleep(2 ** k)


def descargar(condicion):
    registros, ultimo = [], None
    while True:
        cond = f"({condicion})" + (f" AND id_contrato > '{ultimo}'" if ultimo else "")
        pagina = pedir(API, {"$select": ",".join(CAMPOS), "$where": cond, "$order": "id_contrato ASC", "$limit": 5000})
        if not pagina:
            break
        ids = [p["id_contrato"] for p in pagina]
        assert len(ids) == len(set(ids)) and (ultimo is None or min(ids) > ultimo)
        registros += pagina
        ultimo = ids[-1]
        if len(pagina) < 5000:
            break
    return registros


if DESCARGAR_NUEVA:
    fin_excl = (pd.Timestamp(FECHA_CORTE) + pd.Timedelta(days=1)).strftime("%Y-%m-%dT00:00:00")
    cond = (f"nit_entidad IN ({','.join(CATALOGO['nit_entidad'])}) AND fecha_de_firma >= '{FECHA_INICIO}T00:00:00' "
            f"AND fecha_de_firma < '{fin_excl}'")
    ids_antes = {r["id_contrato"] for r in descargar(cond)}
    regs = descargar(cond)
    if {r["id_contrato"] for r in regs} != ids_antes:
        raise RuntimeError("SECOP cambió durante la descarga; repita.")
    id_corrida = f"{DATASET}_{FECHA_CORTE.replace('-', '')}_{pd.Timestamp.now(tz='UTC'):%Y%m%dT%H%M%SZ}"
    carpeta = RUTA_BRUTOS / id_corrida
    carpeta.mkdir(parents=True, exist_ok=False)  # nunca sobrescribe
    with (carpeta / "secop_ii_contratos.jsonl").open("w", encoding="utf-8") as f:
        for r in regs:
            f.write(json.dumps(r, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n")
    su.escribir_json(carpeta / "manifiesto_descarga.json", {
        "id_corrida": id_corrida, "estado": "VALIDADO", "consulta": cond,
        "archivos": {"raw_jsonl": {"ruta_relativa": su.rel(carpeta / "secop_ii_contratos.jsonl", RAIZ),
                                   "sha256": su.sha256_archivo(carpeta / "secop_ii_contratos.jsonl")}}})
    print("Nueva corrida bruta:", id_corrida, len(regs))

## Selección y verificación de la corrida bruta

In [4]:
corridas = []
for man_path in sorted(RUTA_BRUTOS.glob("*/manifiesto_descarga.json")):
    m = su.leer_json(man_path)
    if m.get("estado") == "VALIDADO":
        corridas.append((m["id_corrida"], man_path, m))
if not corridas:
    raise FileNotFoundError("No hay corridas brutas validadas en datos/brutos/secop_ii/corridas.")
ID_CORRIDA, RUTA_MAN_BRUTO, MAN_BRUTO = sorted(corridas, key=lambda x: x[0])[-1]  # la más reciente por ID (incluye timestamp)
RUTA_JSONL = su.abs_desde_rel(MAN_BRUTO["archivos"]["raw_jsonl"]["ruta_relativa"], RAIZ)
if su.sha256_archivo(RUTA_JSONL) != MAN_BRUTO["archivos"]["raw_jsonl"]["sha256"]:
    raise RuntimeError("El JSONL bruto no coincide con su manifiesto: no se usa.")
registros = [json.loads(l) for l in RUTA_JSONL.open(encoding="utf-8") if l.strip()]
print(f"Corrida: {ID_CORRIDA} | registros: {len(registros):,} | hash verificado")

Corrida: jbjy-vk9h_20260906_20260911T205559031273Z | registros: 37,574 | hash verificado


## Base inicial

In [5]:
bruto = pd.DataFrame(registros)
base = pd.DataFrame({
    "id_contrato": bruto["id_contrato"].astype("string").str.strip(),
    "proceso_de_compra": bruto.get("proceso_de_compra"),
    "nit_entidad": bruto["nit_entidad"].astype("string").str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True),
    "nombre_entidad_secop": bruto["nombre_entidad"],
    "ciudad_secop": bruto.get("ciudad"),
    "estado_contrato": bruto["estado_contrato"],
    "tipo_de_contrato": bruto["tipo_de_contrato"],
    "modalidad_de_contratacion": bruto["modalidad_de_contratacion"],
    "justificacion_modalidad": bruto["justificacion_modalidad_de"],
    "descripcion_del_proceso": bruto["descripcion_del_proceso"],
    "tipodocproveedor": bruto["tipodocproveedor"],
    "documento_proveedor": bruto["documento_proveedor"],
    "proveedor_adjudicado": bruto["proveedor_adjudicado"],
    "es_grupo_secop": bruto["es_grupo"],
    "fecha_firma": pd.to_datetime(bruto["fecha_de_firma"], errors="coerce").dt.normalize(),
    "fecha_inicio": pd.to_datetime(bruto["fecha_de_inicio_del_contrato"], errors="coerce").dt.normalize(),
    "fecha_fin": pd.to_datetime(bruto["fecha_de_fin_del_contrato"], errors="coerce").dt.normalize(),
    "valor_contrato": pd.to_numeric(bruto["valor_del_contrato"], errors="coerce"),
    "valor_pagado": pd.to_numeric(bruto["valor_pagado"], errors="coerce"),
    "dias_adicionados": pd.to_numeric(bruto["dias_adicionados"], errors="coerce"),
    "url_secop": bruto["urlproceso"].map(su.extraer_url),
})
base = base.merge(CATALOGO, on="nit_entidad", how="left", validate="many_to_one")
base["fecha_referencia"] = base["fecha_firma"].combine_first(base["fecha_inicio"])
base = base.sort_values("id_contrato", kind="stable").reset_index(drop=True)

ctl = su.Controles()
ctl.agregar("IDs únicos y no vacíos", int(base["id_contrato"].isna().sum() + base["id_contrato"].duplicated().sum()), 0)
ctl.agregar("NIT fuera del catálogo", int(base["entidad"].isna().sum()), 0)
ctl.agregar("Fecha de referencia nula", int(base["fecha_referencia"].isna().sum()), 0)
ctl.agregar("Filas = registros brutos", len(base), len(registros))
fuera = ~base["fecha_referencia"].between(pd.Timestamp(FECHA_INICIO), pd.Timestamp(FECHA_CORTE))
ctl.agregar("Fecha de referencia fuera de ventana", int(fuera.sum()), 0)
ctl.tabla()

,prueba,resultado,esperado,severidad,pasa
0,IDs únicos y no vacíos,0,0,Crítica,True
1,NIT fuera del catálogo,0,0,Crítica,True
2,Fecha de referencia nula,0,0,Crítica,True
3,Filas = registros brutos,37574,37574,Crítica,True
4,Fecha de referencia fuera de ventana,0,0,Crítica,True


## Cobertura observada por entidad y mes
El primer mes con registros de cada entidad es un hecho observado; no prueba exhaustividad.

In [6]:
cob = (base.assign(mes=base["fecha_firma"].dt.to_period("M").dt.to_timestamp())
       .groupby(["entidad", "grupo_atribucion", "mes"]).size().rename("contratos").reset_index())
primeros = (base.groupby(["entidad", "grupo_atribucion"])
            .agg(contratos=("id_contrato", "size"), primera_firma=("fecha_firma", "min"), ultima_firma=("fecha_firma", "max"))
            .reset_index())
central_2021 = cob.loc[cob["entidad"].str.startswith("Alcaldía") & cob["mes"].dt.year.eq(2021)]
print("Alcaldía central, contratos por mes de 2021 (inicio de uso de SECOP II):")
print(central_2021[["mes", "contratos"]].to_string(index=False))
primeros

Alcaldía central, contratos por mes de 2021 (inicio de uso de SECOP II):
       mes  contratos
2021-01-01          1
2021-03-01          2
2021-04-01        139
2021-05-01        408
2021-06-01        287
2021-07-01        306
2021-08-01        518
2021-09-01        604
2021-10-01        353
2021-11-01        336
2021-12-01         81


,entidad,grupo_atribucion,contratos,primera_firma,ultima_firma
0,Alcaldía Distrital de Barrancabermeja,Administración central,27930,2020-02-13,2026-09-04
1,Concejo de Barrancabermeja,No atribuible al alcalde,1546,2021-07-02,2026-09-04
2,Contraloría de Barrancabermeja,No atribuible al alcalde,373,2021-01-22,2026-08-14
3,EDUBA (desarrollo urbano y vivienda),Sector descentralizado vinculado,870,2020-05-07,2026-09-01
4,ESE Barrancabermeja,Sector descentralizado vinculado,2327,2022-07-25,2026-09-02
5,Hospital Regional del Magdalena Medio,No atribuible al alcalde,611,2025-06-19,2026-08-28
6,INDERBA (deporte y recreación),Sector descentralizado vinculado,2489,2021-06-25,2026-09-04
7,Inspección de Tránsito y Transporte,Sector descentralizado vinculado,769,2021-10-25,2026-09-04
8,Personería de Barrancabermeja,No atribuible al alcalde,659,2021-10-07,2026-09-03


## Verificaciones externas (opcionales)
1. ¿Hay otras entidades con ciudad Barrancabermeja fuera del catálogo?
2. ¿Cuántos contratos registró la Alcaldía en SECOP I durante 2021? (si existen, 2021 en SECOP II está incompleto).

In [7]:
verificaciones = []
contratos_secop_i_2021 = None


def registrar(prueba, estado, detalle):
    verificaciones.append({"prueba": prueba, "estado": estado, "detalle": str(detalle)[:500]})


if VERIFICAR_COBERTURA_EXTERNA:
    try:
        ent = pd.DataFrame(pedir(API, {"$select": "nit_entidad,nombre_entidad,count(*) AS n",
                                       "$where": "upper(ciudad) like '%BARRANCABERMEJA%'",
                                       "$group": "nit_entidad,nombre_entidad", "$limit": 500}, timeout=20, intentos=1))
        ent["en_catalogo"] = ent["nit_entidad"].astype(str).str.replace(r"\D", "", regex=True).isin(CATALOGO["nit_entidad"])
        su.guardar_csv(ent, SALIDA / "verificacion_entidades_barrancabermeja.csv")
        registrar("entidades_ciudad_barrancabermeja", "VERIFICADO", f"{int((~ent['en_catalogo']).sum())} NIT fuera del catálogo")
    except Exception as e:
        registrar("entidades_ciudad_barrancabermeja", "NO_VERIFICADO", e)
    try:
        # SECOP I - Procesos de compra pública (f789-7hwg). Se validan los campos antes de consultar.
        meta = pedir("https://www.datos.gov.co/api/views/f789-7hwg", {}, timeout=20, intentos=1)
        campos = {c.get("fieldName") for c in meta.get("columns", [])}
        necesarios = {"nit_de_la_entidad", "fecha_de_firma_del_contrato"}
        if not necesarios <= campos:
            registrar("secop_i_alcaldia_2021", "CAMPOS_NO_ENCONTRADOS", sorted(necesarios - campos))
        else:
            r = pedir("https://www.datos.gov.co/resource/f789-7hwg.json", {
                "$select": "date_extract_m(fecha_de_firma_del_contrato) AS mes, count(*) AS n",
                "$where": f"nit_de_la_entidad='{su.NIT_ALCALDIA}' AND fecha_de_firma_del_contrato between "
                          "'2021-01-01T00:00:00' and '2021-12-31T23:59:59'",
                "$group": "mes", "$order": "mes"}, timeout=30, intentos=1)
            tabla = pd.DataFrame(r)
            su.guardar_csv(tabla, SALIDA / "verificacion_secop_i_alcaldia_2021.csv")
            total = int(pd.to_numeric(tabla.get("n", pd.Series(dtype=str))).sum()) if len(tabla) else 0
            contratos_secop_i_2021 = total
            registrar("secop_i_alcaldia_2021", "VERIFICADO", f"{total} contratos SECOP I firmados en 2021")
    except Exception as e:
        registrar("secop_i_alcaldia_2021", "NO_VERIFICADO", e)
else:
    registrar("entidades_ciudad_barrancabermeja", "NO_VERIFICADO", "desactivado")
    registrar("secop_i_alcaldia_2021", "NO_VERIFICADO", "desactivado")
verificaciones = pd.DataFrame(verificaciones)
# 2021 solo se considera completo si SECOP I confirma cero contratos de la Alcaldía ese año.
COBERTURA_2021_VALIDADA = contratos_secop_i_2021 == 0
print("Cobertura 2021 validada como completa:", COBERTURA_2021_VALIDADA)
verificaciones

Cobertura 2021 validada como completa: False


,prueba,estado,detalle
0,entidades_ciudad_barrancabermeja,NO_VERIFICADO,Consulta fallida: <urlopen error Tunnel connec...
1,secop_i_alcaldia_2021,NO_VERIFICADO,Consulta fallida: <urlopen error Tunnel connec...


## Escritura y cierre

In [8]:
ruta_estado = SALIDA / "estado_cobertura.json"
su.escribir_json(ruta_estado, {"cobertura_2021_validada": COBERTURA_2021_VALIDADA,
                               "contratos_secop_i_2021": contratos_secop_i_2021,
                               "primer_mes_central_50_o_mas": str(central_2021.loc[central_2021["contratos"].ge(50), "mes"].min().date())})
salidas = {
    "estado_cobertura": ruta_estado,
    "base_inicial": su.guardar_csv(base, SALIDA / "base_inicial.csv"),
    "cobertura_mensual": su.guardar_csv(cob, SALIDA / "cobertura_mensual_entidad.csv"),
    "resumen_entidades": su.guardar_csv(primeros, SALIDA / "resumen_entidades.csv"),
    "verificaciones_externas": su.guardar_csv(verificaciones, SALIDA / "verificaciones_externas.csv"),
    "controles": su.guardar_csv(ctl.tabla(), SALIDA / "controles_01.csv"),
    "catalogo": ruta_catalogo,
}
estado = "BLOQUEADO" if ctl.bloqueos() else "VALIDADO"
man = su.cerrar_etapa(
    RAIZ, ETAPA, VERSION_NB,
    entradas={"corrida_bruta": ID_CORRIDA, "sha256_jsonl": MAN_BRUTO["archivos"]["raw_jsonl"]["sha256"]},
    salidas=salidas,
    reglas={"atribucion": "Solo Alcaldía central y sector descentralizado vinculado dependen del alcalde; Concejo, órganos de control y Hospital Regional no.",
            "fechas": "fecha_firma normalizada al día; sin imputación."},
    conteos={"contratos": len(base), "entidades": int(base["entidad"].nunique())},
    estado=estado,
    alertas=ctl.bloqueos() + [f"{r.prueba}: {r.estado}" for r in verificaciones.itertuples() if r.estado != "VERIFICADO"],
)
if ctl.bloqueos():
    raise RuntimeError(f"Etapa 01 bloqueada: {ctl.bloqueos()}")
print(man["estado"], "| alertas:", man["alertas"])

VALIDADO | alertas: ['entidades_ciudad_barrancabermeja: NO_VERIFICADO', 'secop_i_alcaldia_2021: NO_VERIFICADO']
